In [1]:
import pandas as pd
import numpy as np

In [2]:
# we'll use the catalog as a stand-in for "user's furniture" while testing
catalog = pd.read_json("catalog.json")
catalog = catalog.dropna(subset=["width_mm", "depth_mm", "height_mm"]).reset_index(drop=True)

## Fit Check

In [3]:
def check_fit(items, room_w, room_d, room_h, usable_fraction=0.6):
    """Does the user's furniture fit a new room?
    Returns a dict with the result + details."""

    # 1. anything too tall for the ceiling?
    too_tall = items[items["height_mm"] > room_h]

    # 2. any single item too big to fit through the floor (both rotations)?
    fit_a = (items["width_mm"] <= room_w) & (items["depth_mm"] <= room_d)
    fit_b = (items["width_mm"] <= room_d) & (items["depth_mm"] <= room_w)
    too_big = items[~(fit_a | fit_b)]

    # 3. total footprint vs usable floor
    needed = (items["width_mm"] * items["depth_mm"]).sum()
    available = room_w * room_d * usable_fraction

    fits = (len(too_tall) == 0) and (len(too_big) == 0) and (needed <= available)

    return {
        "fits": fits,
        "needed_m2": round(needed / 1e6, 2),
        "available_m2": round(available / 1e6, 2),
        "too_tall": too_tall,
        "too_big": too_big,
    }

In [8]:
# pretend the user owns these 5 items
user_items = catalog.sample(5)
print("User's furniture:")
user_items[["name", "category", "width_mm", "depth_mm", "height_mm"]]

User's furniture:


,name,category,width_mm,depth_mm,height_mm
283,Ravenna Home Classic Console End Table with Si...,table,1067.0,406.0,762.0
141,"Rivet York Modern Extended Platform Queen Bed,...",bed,2290.0,1829.0,1075.0
13,"AmazonBasics Fauteuil de bureau et de gaming, ...",chair,730.0,550.0,1260.0
11,"Amazon Brand - Movian Ribe - Divano a 3 posti,...",sofa,2000.0,920.0,850.0
202,Amazon Brand – Stone & Beam Kristin Round Arm ...,sofa,1930.0,940.0,952.0


In [9]:
# try fitting them in a small student room
result = check_fit(user_items, room_w=3000, room_d=2500, room_h=2400)

print(f"Fits? {result['fits']}")
print(f"Needed: {result['needed_m2']} m²   Available: {result['available_m2']} m²")

if len(result["too_tall"]) > 0:
    print("\nToo tall for ceiling:")
    print(result["too_tall"][["name", "height_mm"]])
if len(result["too_big"]) > 0:
    print("\nWon't fit on floor:")
    print(result["too_big"][["name", "width_mm", "depth_mm"]])

Fits? False
Needed: 8.68 m²   Available: 4.5 m²


## Greedy Approach to fit items

In [10]:
def suggest_drops(items, room_w, room_d, usable_fraction=0.6):
    """If the items don't fit, find the smallest set to remove so they do.
    Greedy: drop the biggest item first, keep going until it fits."""
    available = room_w * room_d * usable_fraction

    # sort biggest footprint first
    items = items.copy()
    items["footprint"] = items["width_mm"] * items["depth_mm"]
    items = items.sort_values("footprint", ascending=False).reset_index(drop=True)

    dropped = []
    keep = items.copy()
    while keep["footprint"].sum() > available and len(keep) > 0:
        dropped.append(keep.iloc[0])
        keep = keep.iloc[1:]

    return pd.DataFrame(dropped), keep

In [11]:
to_drop, to_keep = suggest_drops(user_items, 3000, 2500)

print("Drop these to make it fit:")
print(to_drop[["name", "category", "width_mm", "depth_mm"]])

print("\nKeep these:")
print(to_keep[["name", "category", "width_mm", "depth_mm"]])

Drop these to make it fit:
                                                name category  width_mm  \
0  Rivet York Modern Extended Platform Queen Bed,...      bed    2290.0   

   depth_mm  
0    1829.0  

Keep these:
                                                name category  width_mm  \
1  Amazon Brand - Movian Ribe - Divano a 3 posti,...     sofa    2000.0   
2  Amazon Brand – Stone & Beam Kristin Round Arm ...     sofa    1930.0   
3  Ravenna Home Classic Console End Table with Si...    table    1067.0   
4  AmazonBasics Fauteuil de bureau et de gaming, ...    chair     730.0   

   depth_mm  
1     920.0  
2     940.0  
3     406.0  
4     550.0  
